In [311]:
import importlib
importlib.reload(kafi.streams.topologynode)

import kafi.streams.topologynode
from kafi.streams.topologynode import source

import cloudpickle as pickle

#

order_source_str = "orders"
order_source_tn = source(order_source_str)
order_with_window_end_tn = (
    order_source_tn
    .map(lambda x: x | {"window_end": x["value"]["ts"] + 10})
)
order_cur_ts_tn = (
    order_source_tn
    .map(lambda x: {"ts": x["value"]["ts"]})
    .max(lambda x: x["ts"],
         lambda x: {"cur_ts": x})
)
join_window_end_tn = (
    order_with_window_end_tn
    .join(order_cur_ts_tn,
          lambda l, r: r["cur_ts"] > l["window_end"],
          lambda l, _: l)
    .neg()
    ._delay()
)
root_tn = (
    join_window_end_tn
    .union(order_with_window_end_tn)
    ._differentiate()
    ._peek()
)
root_tn.build()
order_source_tn._to_zSet_function = root_tn.record_any_weight_int_tuple_list_to_zSet
join_window_end_tn._from_zSet_function = root_tn.zSet_to_record_any_weight_int_tuple_list
root_tn._from_zSet_function = root_tn.zSet_to_record_any_weight_int_tuple_list
#
def gen_order(order_id_int, ts_int):
    return {"value": {"order_id": order_id_int, "ts": ts_int}}
#
print("Step 1.1")
root_tn.push(order_source_str, [(gen_order(1, 0), 1)])
root_tn.latest()
print(len(pickle.dumps(root_tn)))
print("Step 1.2")
root_tn.push(order_source_str, join_window_end_tn.latest())
root_tn.latest()
print(len(pickle.dumps(root_tn)))
print("Step 2.1")
root_tn.push(order_source_str, [(gen_order(2, 11), 1)])
root_tn.latest()
print(len(pickle.dumps(root_tn)))
print("Step 2.2")
root_tn.push(order_source_str, join_window_end_tn.latest())
root_tn.latest()
print(len(pickle.dumps(root_tn)))
print("Step 3.1")
root_tn.push(order_source_str, [(gen_order(3, 0), 1)])
root_tn.latest()
print(len(pickle.dumps(root_tn)))
print("Step 3.2")
root_tn.push(order_source_str, join_window_end_tn.latest())
root_tn.latest()
print(len(pickle.dumps(root_tn)))

for i in range(100):
    root_tn.push(order_source_str, [(gen_order(i + 4, i * 10 + 11), 1)])
    root_tn.latest()
    root_tn.push(order_source_str, join_window_end_tn.latest())
    root_tn.latest()
    print(len(pickle.dumps(root_tn)))



Step 1.1
({'value': {'order_id': 1, 'ts': 0}, 'window_end': 10}, 1)
24576
Step 1.2
25870
Step 2.1
({'value': {'order_id': 2, 'ts': 11}, 'window_end': 21}, 1)
26921
Step 2.2
({'value': {'order_id': 1, 'ts': 0}, 'window_end': 10}, -1)
27930
Step 3.1
({'value': {'order_id': 3, 'ts': 0}, 'window_end': 10}, 1)
28700
Step 3.2
({'value': {'order_id': 3, 'ts': 0}, 'window_end': 10}, -1)
29267
({'value': {'order_id': 4, 'ts': 11}, 'window_end': 21}, 1)
29407
({'value': {'order_id': 5, 'ts': 21}, 'window_end': 31}, 1)
29187
({'value': {'order_id': 6, 'ts': 31}, 'window_end': 41}, 1)
({'value': {'order_id': 4, 'ts': 11}, 'window_end': 21}, -1)
({'value': {'order_id': 2, 'ts': 11}, 'window_end': 21}, -1)
29580
({'value': {'order_id': 7, 'ts': 41}, 'window_end': 51}, 1)
({'value': {'order_id': 5, 'ts': 21}, 'window_end': 31}, -1)
30048
({'value': {'order_id': 8, 'ts': 51}, 'window_end': 61}, 1)
({'value': {'order_id': 6, 'ts': 31}, 'window_end': 41}, -1)
30015
({'value': {'order_id': 9, 'ts': 61}, 